In [1]:
import sympy as sp

### Symbols

In [2]:
omega = sp.symbols('omega', real=True)  # Frequency variable

g_t = sp.symbols('g_t') # Transduction coupling rate (complex)
g_s = sp.symbols('g_s') # Squeezing coupling rate (complex)

Delta = sp.symbols('Delta', real=True)      # Optical frequency detuning
omega_b = sp.symbols('omega_b', real=True)  # Microwave frequency

kappa_a = sp.symbols('kappa_a', real=True)  # Optical decay rate
kappa_b = sp.symbols('kappa_b', real=True)  # Microwave decay rate

kappa_a_ext = sp.symbols('kappa_a^ext', real=True, nonnegative=True)  # Optical external coupling rate
kappa_b_ext = sp.symbols('kappa_b^ext', real=True, nonnegative=True)  # Microwave external coupling rate

kappa_a_int = sp.symbols('kappa_a^int', real=True, nonnegative=True)  # Optical internal loss rate
kappa_b_int = sp.symbols('kappa_b^int', real=True, nonnegative=True)  # Microwave internal loss rate

kappa_a_tilde = sp.symbols('\\tilde{\\kappa}_a')  # Frequency-dependent complex optical decay rate
kappa_b_tilde = sp.symbols('\\tilde{\\kappa}_b')  # Frequency-dependent complex microwave decay rate

kappa_a_tilde_p = sp.symbols('\\tilde{\\kappa}_a^+')  # Frequency-dependent complex optical decay rate
kappa_b_tilde_p = sp.symbols('\\tilde{\\kappa}_b^+')  # Frequency-dependent complex microwave decay rate

kappa_a_tilde_m = sp.symbols('\\tilde{\\kappa}_a^-')  # Frequency-dependent complex optical decay rate
kappa_b_tilde_m = sp.symbols('\\tilde{\\kappa}_b^-')  # Frequency-dependent complex microwave decay rate

### Functions

In [3]:
def diag_sqrt(M):
    """
    Square root of a diagonal matrix — takes sqrt of each diagonal entry.
    """
    return sp.diag(*[sp.sqrt(M[i, i]) for i in range(M.shape[0])])

### Matrices

In [4]:
A = sp.Matrix([
    [-sp.I*Delta - kappa_a/2,   sp.I * g_t,                 0,                      sp.I * g_s         ],
    [sp.I * sp.conjugate(g_t),  -sp.I*omega_b - kappa_b/2,  sp.I * g_s,             0           ],
    [0,                         -sp.I * sp.conjugate(g_s),  sp.I*Delta - kappa_a/2, -sp.I * sp.conjugate(g_t)],
    [-sp.I * sp.conjugate(g_s), 0,                          -sp.I * g_t,            sp.I*omega_b - kappa_b/2],
])

display(A)

Matrix([
[-I*Delta - kappa_a/2,                  I*g_t,                   0,                  I*g_s],
[    I*conjugate(g_t), -kappa_b/2 - I*omega_b,               I*g_s,                      0],
[                   0,      -I*conjugate(g_s), I*Delta - kappa_a/2,      -I*conjugate(g_t)],
[   -I*conjugate(g_s),                      0,              -I*g_t, -kappa_b/2 + I*omega_b]])

In [5]:
K_ext = sp.diag(kappa_a_ext, kappa_b_ext, kappa_a_ext, kappa_b_ext)
K_int = sp.diag(kappa_a_int, kappa_b_int, kappa_a_int, kappa_b_int)

K_ext

Matrix([
[kappa_a^ext,           0,           0,           0],
[          0, kappa_b^ext,           0,           0],
[          0,           0, kappa_a^ext,           0],
[          0,           0,           0, kappa_b^ext]])

In [6]:
S = - sp.eye(4) - sp.sqrt(K_ext) * (A + sp.I * omega * sp.eye(4)).inv('ADJ') * sp.sqrt(K_ext)

S

Matrix([
[-kappa_a^ext*(I*Delta*kappa_b**2/4 + Delta*kappa_b*omega - I*Delta*omega**2 + I*Delta*omega_b**2 + g_s*kappa_b*conjugate(g_s)/2 - I*g_s*omega*conjugate(g_s) - I*g_s*omega_b*conjugate(g_s) - g_t*kappa_b*conjugate(g_t)/2 + I*g_t*omega*conjugate(g_t) - I*g_t*omega_b*conjugate(g_t) - kappa_a*kappa_b**2/8 + I*kappa_a*kappa_b*omega/2 + kappa_a*omega**2/2 - kappa_a*omega_b**2/2 + I*kappa_b**2*omega/4 + kappa_b*omega**2 - I*omega**3 + I*omega*omega_b**2)/(-g_s*(-kappa_b/2 + I*omega + I*omega_b)**2*conjugate(g_s) + g_t*(-kappa_b/2 + I*omega - I*omega_b)**2*conjugate(g_t) + (-g_s*conjugate(g_s) + g_t*conjugate(g_t))*(-g_s*conjugate(g_s) + g_t*conjugate(g_t) + (kappa_b/2 - I*omega - I*omega_b)*(-I*Delta + kappa_a/2 - I*omega) + (kappa_b/2 - I*omega + I*omega_b)*(-I*Delta + kappa_a/2 + kappa_b/2 - 2*I*omega - I*omega_b)) + (-g_s*(-kappa_b/2 + I*omega + I*omega_b)*conjugate(g_s) + g_t*(-kappa_b/2 + I*omega - I*omega_b)*conjugate(g_t))*(-I*Delta + kappa_a/2 + kappa_b - 3*I*omega) + (I*Delt

In [7]:
Sp = sp.zeros(S.shape[0], S.shape[1])
for i in [0]:
    for j in [0, 1, 2, 3]:
        s = S[i, j]
        
        # First pass: substitute in the frequency-dependent decay rates.
        s = sp.simplify(s.subs(kappa_a, kappa_a_tilde + 2 * sp.I * omega))
        s = sp.simplify(s.subs(kappa_b, kappa_b_tilde + 2 * sp.I * omega))

        # Second pass: substitute in the single-photon coupling rate.
        s = sp.simplify(s.subs(sp.conjugate(g_t), sp.Abs(g_t)**2/g_t))
        s = sp.simplify(s.subs(sp.conjugate(g_s), sp.Abs(g_s)**2/g_s))

        # Fourth pass: subsititute in plus/minus forms of the frequency-dependent decay rates.
        s = sp.simplify(s.subs(kappa_a_tilde, (kappa_a_tilde_p + kappa_a_tilde_m)/2))
        s = sp.simplify(s.subs(kappa_b_tilde, (kappa_b_tilde_p + kappa_b_tilde_m)/2))
        s = sp.simplify(s.subs(Delta, (kappa_a_tilde_p - kappa_a_tilde_m)/(4*sp.I)))
        s = sp.simplify(s.subs(omega_b, (kappa_b_tilde_p - kappa_b_tilde_m)/(4*sp.I)))

        s = sp.simplify(s)
        Sp[i, j] = s

### Raw forms for $S_{1x}$

In [8]:
Sp[0, 0]

(-\tilde{\kappa}_a^+*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^- - 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^+*Abs(g_t)**2 + 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^-*Abs(g_s)**2 + 2*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^-*kappa_a^ext + 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*Abs(g_s)**2 - 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^-*Abs(g_t)**2 + 8*\tilde{\kappa}_b^+*kappa_a^ext*Abs(g_t)**2 - 8*\tilde{\kappa}_b^-*kappa_a^ext*Abs(g_s)**2 - 16*Abs(g_s)**4 - 16*Abs(g_t)**4 + 32*Abs(g_s**2*g_t**2))/(\tilde{\kappa}_a^+*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^- + 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^+*Abs(g_t)**2 - 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^-*Abs(g_s)**2 - 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*Abs(g_s)**2 + 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^-*Abs(g_t)**2 + 16*Abs(g_s)**4 + 16*Abs(g_t)**4 - 32*Abs(g_s**2*g_t**2))

In [9]:
Sp[0, 1]

4*I*g_t*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)*(\tilde{\kappa}_a^-*\tilde{\kappa}_b^- - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)/(\tilde{\kappa}_a^+*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^- + 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^+*Abs(g_t)**2 - 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^-*Abs(g_s)**2 - 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*Abs(g_s)**2 + 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^-*Abs(g_t)**2 + 16*Abs(g_s)**4 + 16*Abs(g_t)**4 - 32*Abs(g_s**2*g_t**2))

In [10]:
Sp[0, 2]

8*g_s*g_t*kappa_a^ext*(\tilde{\kappa}_b^+ - \tilde{\kappa}_b^-)/(\tilde{\kappa}_a^+*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^- + 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^+*Abs(g_t)**2 - 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^-*Abs(g_s)**2 - 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*Abs(g_s)**2 + 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^-*Abs(g_t)**2 + 16*Abs(g_s)**4 + 16*Abs(g_t)**4 - 32*Abs(g_s**2*g_t**2))

In [11]:
Sp[0, 3]

4*I*g_s*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)*(\tilde{\kappa}_a^-*\tilde{\kappa}_b^+ - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)/(\tilde{\kappa}_a^+*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^- + 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^+*Abs(g_t)**2 - 4*\tilde{\kappa}_a^+*\tilde{\kappa}_b^-*Abs(g_s)**2 - 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*Abs(g_s)**2 + 4*\tilde{\kappa}_a^-*\tilde{\kappa}_b^-*Abs(g_t)**2 + 16*Abs(g_s)**4 + 16*Abs(g_t)**4 - 32*Abs(g_s**2*g_t**2))

### Simplified forms for $S_{12}$ and $S_{14}$

In [12]:
numer_S12, denom_S12 = Sp[0, 1].as_numer_denom()

prefactor_S12 = 4 * sp.I * g_t * sp.sqrt(kappa_a_ext * kappa_b_ext)
denom_S12_first_term = (kappa_a_tilde_p * kappa_b_tilde_p + 4 * (sp.Abs(g_t)**2 - sp.Abs(g_s)**2))
denom_S12_second_term = 4 * sp.Abs(g_s)**2 * (kappa_a_tilde_p - kappa_a_tilde_m) * (kappa_b_tilde_p - kappa_b_tilde_m)

denom_S12_guess = (
    (numer_S12 / prefactor_S12)
    * denom_S12_first_term
    + denom_S12_second_term
)

(denom_S12 - denom_S12_guess).simplify().factor()

0

In [13]:
numer_S14, denom_S14 = Sp[0, 3].as_numer_denom()

prefactor_S14 = 4 * sp.I * g_s * sp.sqrt(kappa_a_ext * kappa_b_ext)
denom_S14_first_term = (kappa_a_tilde_p * kappa_b_tilde_m + 4 * (sp.Abs(g_t)**2 - sp.Abs(g_s)**2))
denom_S14_second_term = 4 * sp.Abs(g_t)**2 * (kappa_a_tilde_p - kappa_a_tilde_m) * (kappa_b_tilde_p - kappa_b_tilde_m)

denom_S14_guess = (
    (numer_S14 / prefactor_S14)
    * denom_S14_first_term
    + denom_S14_second_term
)

(denom_S14 - denom_S14_guess).simplify().factor()

0

In [ ]:
simplified_S12 = prefactor_S12 / (
    denom_S12_first_term
    + denom_S12_second_term * (prefactor_S12 / numer_S12)
)

simplified_S12 = simplified_S12.subs(kappa_a_tilde_p, kappa_a_tilde + 2 * sp.I * Delta)
simplified_S12 = simplified_S12.subs(kappa_a_tilde_m, kappa_a_tilde - 2 * sp.I * Delta)
simplified_S12 = simplified_S12.subs(kappa_b_tilde_p, kappa_b_tilde + 2 * sp.I * omega_b)
simplified_S12 = simplified_S12.subs(kappa_b_tilde_m, kappa_b_tilde - 2 * sp.I * omega_b)

simplified_S12

4*I*g_t*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)/(\tilde{\kappa}_a^+*\tilde{\kappa}_b^+ + 4*(\tilde{\kappa}_a^+ - \tilde{\kappa}_a^-)*(\tilde{\kappa}_b^+ - \tilde{\kappa}_b^-)*Abs(g_s)**2/(\tilde{\kappa}_a^-*\tilde{\kappa}_b^- - 4*Abs(g_s)**2 + 4*Abs(g_t)**2) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)

In [ ]:
simplified_S14 = prefactor_S14 / (
    denom_S14_first_term
    + denom_S14_second_term * (prefactor_S14 / numer_S14)
)

simplified_S14 = simplified_S14.subs(kappa_a_tilde_p, kappa_a_tilde + 2 * sp.I * Delta)
simplified_S14 = simplified_S14.subs(kappa_a_tilde_m, kappa_a_tilde - 2 * sp.I * Delta)
simplified_S14 = simplified_S14.subs(kappa_b_tilde_p, kappa_b_tilde + 2 * sp.I * omega_b)
simplified_S14 = simplified_S14.subs(kappa_b_tilde_m, kappa_b_tilde - 2 * sp.I * omega_b)

simplified_S14

4*I*g_s*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)/(\tilde{\kappa}_a^+*\tilde{\kappa}_b^- + 4*(\tilde{\kappa}_a^+ - \tilde{\kappa}_a^-)*(\tilde{\kappa}_b^+ - \tilde{\kappa}_b^-)*Abs(g_t)**2/(\tilde{\kappa}_a^-*\tilde{\kappa}_b^+ - 4*Abs(g_s)**2 + 4*Abs(g_t)**2) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)

### Forms for the paper

Quote $S_{12}$ in its D2 form, then state the two ratios $S_{13}/S_{12}$ and $S_{14}/S_{12}$. The shared denominator cancels in each ratio, so they render in closed form (no $\mathcal{R}_-$), and they are exactly the quantities to Taylor-expand.

Because $|S_{12}|\to1$ at the operating (unit-transduction) point, each ratio is also the *absolute* size of $S_{13},S_{14}$ there — so the ratio carries all the $\lambda$-scaling while $S_{12}$ supplies the (order-unity) transduction efficiency.

#### $S_{12}$ — transduction (Eq. D2)

In [ ]:
simplified_S12 = simplified_S12.subs(kappa_a_tilde_m * kappa_a_tilde_m, kappa_a_tilde + 2 * sp.I * Delta)

4*I*g_t*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)/(\tilde{\kappa}_a^+*\tilde{\kappa}_b^+ + 4*(\tilde{\kappa}_a^+ - \tilde{\kappa}_a^-)*(\tilde{\kappa}_b^+ - \tilde{\kappa}_b^-)*Abs(g_s)**2/(\tilde{\kappa}_a^-*\tilde{\kappa}_b^- - 4*Abs(g_s)**2 + 4*Abs(g_t)**2) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)

##### $S_{11}/S_{12}$, $S_{13}/S_{12}$, $S_{14}/S_{12}$ to the transduction amplitude

In [17]:
S11 = Sp[0, 0]

numer_S11, denom_S11 = Sp[0, 0].as_numer_denom()

# S11_over_S12 = S11_over_S12.subs(kappa_a_tilde_p, kappa_a_tilde + 2 * sp.I * Delta)
# S11_over_S12 = S11_over_S12.subs(kappa_a_tilde_m, kappa_a_tilde - 2 * sp.I * Delta)
# S11_over_S12 = S11_over_S12.subs(kappa_b_tilde_p, kappa_b_tilde + 2 * sp.I * omega_b)
# S11_over_S12 = S11_over_S12.subs(kappa_b_tilde_m, kappa_b_tilde - 2 * sp.I * omega_b)

# S11_over_S12 = S11_over_S12.simplify()

(numer_S11+denom_S11)/denom_S11/Sp[0, 1].simplify()

-I*(2*\tilde{\kappa}_a^-*\tilde{\kappa}_b^+*\tilde{\kappa}_b^-*kappa_a^ext + 8*\tilde{\kappa}_b^+*kappa_a^ext*Abs(g_t)**2 - 8*\tilde{\kappa}_b^-*kappa_a^ext*Abs(g_s)**2)/(4*g_t*sqrt(kappa_a^ext)*sqrt(kappa_b^ext)*(\tilde{\kappa}_a^-*\tilde{\kappa}_b^- - 4*Abs(g_s)**2 + 4*Abs(g_t)**2))

In [18]:
S13_over_S12 = Sp[0, 2]/Sp[0, 1]

S13_over_S12 = S13_over_S12.subs(kappa_a_tilde_p, kappa_a_tilde + 2 * sp.I * Delta)
S13_over_S12 = S13_over_S12.subs(kappa_a_tilde_m, kappa_a_tilde - 2 * sp.I * Delta)
S13_over_S12 = S13_over_S12.subs(kappa_b_tilde_p, kappa_b_tilde + 2 * sp.I * omega_b)
S13_over_S12 = S13_over_S12.subs(kappa_b_tilde_m, kappa_b_tilde - 2 * sp.I * omega_b)

S13_over_S12

8*g_s*sqrt(kappa_a^ext)*omega_b/(sqrt(kappa_b^ext)*((\tilde{\kappa}_b - 2*I*omega_b)*(-2*I*Delta + \tilde{\kappa}_a) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2))

In [19]:
S14_over_S12 = Sp[0, 3]/Sp[0, 1]

S14_over_S12 = S14_over_S12.subs(kappa_a_tilde_p, kappa_a_tilde + 2 * sp.I * Delta)
S14_over_S12 = S14_over_S12.subs(kappa_a_tilde_m, kappa_a_tilde - 2 * sp.I * Delta)
S14_over_S12 = S14_over_S12.subs(kappa_b_tilde_p, kappa_b_tilde + 2 * sp.I * omega_b)
S14_over_S12 = S14_over_S12.subs(kappa_b_tilde_m, kappa_b_tilde - 2 * sp.I * omega_b)

S14_over_S12

g_s*((\tilde{\kappa}_b + 2*I*omega_b)*(-2*I*Delta + \tilde{\kappa}_a) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2)/(g_t*((\tilde{\kappa}_b - 2*I*omega_b)*(-2*I*Delta + \tilde{\kappa}_a) - 4*Abs(g_s)**2 + 4*Abs(g_t)**2))

#### $S_{14}/S_{12}$ as $1+$fraction (shares the denominator with $S_{13}/S_{12}$)

In [20]:
# S_14/S_12 as 1 + fraction: [k_b-2i(w-w_b')] = [k_b-2i(w+w_b')] + 4i*w_b' regenerates D_shared,
# leaving 1 + fraction (same denominator as S_13/S_12).
ratio_S14_oneplus = (g_s/g_t)*(1 + 4*sp.I*omega_b_p*kappa_a_tilde_m / D_shared)
assert sp.simplify(ratio_S14_oneplus.subs(_wb_sub) - Sp[0, 3]/Sp[0, 1]) == 0
ratio_S14_oneplus

NameError: name 'omega_b_p' is not defined

#### Old-physics recovery: $g_s\to0$ collapses $S_{12}$ onto the transduction resonance (cf. Ref. [22])

In [ ]:
sp.simplify(simplified_S01.subs(g_s, 0))

NameError: name 'simplified_S01' is not defined

#### Band-centre leading order (set $\Delta=\omega_b,\ \omega=\omega_b$; rates$/\omega_b=O(\lambda)$)
Each ratio is $O(\lambda)$ — the off-resonant two-mode-squeezing suppression.

In [ ]:
lam = sp.symbols('lambda', positive=True)
bandcentre = {
    kappa_a_tilde_p: lam*kappa_a,
    kappa_b_tilde_p: lam*kappa_b,
    kappa_a_tilde_m: lam*kappa_a - 4*sp.I*omega_b,
    kappa_b_tilde_m: lam*kappa_b - 4*sp.I*omega_b,
    sp.Abs(g_t): lam*sp.Abs(g_t),
    sp.Abs(g_s): lam*sp.Abs(g_s),
    omega_b_p: omega_b,
}
lead_S14 = sp.simplify(sp.series(ratio_S14_over_S12.subs(bandcentre), lam, 0, 2).removeO())
lead_S14   # -> i*lambda*(g_s/g_t)*kappa_b/(4*omega_b) = O(lambda)

NameError: name 'ratio_S14_over_S12' is not defined

In [ ]:
lead_S13 = sp.simplify(sp.series(ratio_S13_over_S12.subs(bandcentre), lam, 0, 2).removeO())
lead_S13   # dominant term -> -(g_s/2 omega_b) sqrt(kappa_a^ext/kappa_b^ext)

g_s*sqrt(kappa_a^ext)*(I*lambda*(kappa_a + kappa_b) - 4*omega_b)/(8*sqrt(kappa_b^ext)*omega_b**2)

#### $\mathcal{R}$ is $O(\lambda)$, so transduction is protected to $O(\lambda^2)$

In [ ]:
# R(omega') is itself O(lambda): the 16 omega_b'^2 numerator nearly cancels the -16 omega_b'^2 denominator.
R_expr = 1 + 16*omega_b_p**2 / D_shared          # D9, with the (tilde) off-resonant denominator
R_lead = sp.simplify(sp.series(R_expr.subs(bandcentre), lam, 0, 2).removeO())
assert sp.simplify(R_lead - sp.I*lam*(kappa_a + kappa_b)/(4*omega_b)) == 0
R_lead   # -> i*lambda*(kappa_a + kappa_b)/(4*omega_b) = O(lambda)

I*lambda*(kappa_a + kappa_b)/(4*omega_b)

In [ ]:
# S_12 denominator to O(lambda^2): the biphoton term -4|g_s|^2 R is O(lambda^3) and drops,
# leaving the Ref. [22] transduction coefficient  4|g_t|^2 + kappa_a~ kappa_b~.
den_S12 = 4*sp.Abs(g_t)**2 - 4*sp.Abs(g_s)**2*R_expr + kappa_a_tilde_p*kappa_b_tilde_p
den_S12_O2 = sp.simplify(sp.series(den_S12.subs(bandcentre), lam, 0, 3).removeO())
assert sp.simplify(den_S12_O2 - lam**2*(kappa_a*kappa_b + 4*sp.Abs(g_t)**2)) == 0
den_S12_O2   # -> lambda^2 (kappa_a kappa_b + 4|g_t|^2) : Ref. [22], the R term is gone

lambda**2*(kappa_a*kappa_b + 4*Abs(g_t)**2)

#### $O(\lambda^2)$ expansions D11–D13, pinned (residual must fall as $\lambda^3$)

In [ ]:
import numpy as np

# O(lambda^2) expansions D11-D13, pinned by a numeric order-check: the residual against the
# exact Sp entries must fall like lambda^3 (i.e. the closed forms are correct through O(lambda^2)).
def _resid_D11_D13(lam, seed=1):
    rng = np.random.default_rng(seed); wb = 1.0
    ka, kb = lam*rng.uniform(.3, 1.2), lam*rng.uniform(.3, 1.2)
    gt = lam*(rng.uniform(.2, .5) + 1j*rng.uniform(-.2, .2))
    gs = lam*(rng.uniform(.05, .2) + 1j*rng.uniform(-.1, .1))
    kaext, kbext = rng.uniform(.3, .8), rng.uniform(.3, .8)
    sub = {kappa_a_tilde_p: ka, kappa_a_tilde_m: ka - 4j*wb,
           kappa_b_tilde_p: kb, kappa_b_tilde_m: kb - 4j*wb,
           g_t: gt, g_s: gs, kappa_a_ext: kaext, kappa_b_ext: kbext}
    S12 = complex(Sp[0, 1].subs(sub)); S13 = complex(Sp[0, 2].subs(sub)); S14 = complex(Sp[0, 3].subs(sub))
    R = 1 + 16*wb**2 / (4*abs(gt)**2 - 4*abs(gs)**2 + (ka - 4j*wb)*(kb - 4j*wb))
    sq = np.sqrt(kaext/kbext)
    # D11, D12, D13 exactly as written in the paper (kappa~ = kappa at omega'=0):
    D11 = -(gs/(2*wb))*sq*(1 + (ka + kb)/(4j*wb))
    D12 = -(gs/gt)*(kb/(4j*wb) - kb**2/(16*wb**2) + abs(gt)**2/(4*wb**2) - abs(gs)**2/(4*wb**2))
    D13 = -(ka + kb)/(4j*wb) - abs(gt)**2/(4*wb**2) + abs(gs)**2/(4*wb**2) + (ka**2 + ka*kb + kb**2)/(16*wb**2)
    return abs(S13/S12 - D11), abs(S14/S12 - D12), abs(R - D13)

_ea = _resid_D11_D13(0.06); _eb = _resid_D11_D13(0.02)
for _x, _y, _n in zip(_ea, _eb, ['D11', 'D12', 'D13']):
    _order = np.log(_x/_y)/np.log(0.06/0.02)
    assert 2.7 < _order < 3.3, (_n, _order)   # residual ~ lambda^3  =>  correct to O(lambda^2)
print('D11, D12, D13 verified to O(lambda^2)  (residual ~ lambda^3)')

D11, D12, D13 verified to O(lambda^2)  (residual ~ lambda^3)
